# Filtering data from yfinance

In [10]:
%pip install yfinance

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\Wiltj\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [11]:
import pandas as pd
import warnings

warnings.filterwarnings(
    "ignore", category=FutureWarning, message=".*un-recognized timezone.*"
)


Getting data from yfinance starting from the dates that we have gotten from the news data

In [18]:
import yfinance as yf

start_date = "2017-12-17"
end_date = "2020-07-18"

raw_data = yf.download(tickers = "^GSPC", start = start_date,
                              end = end_date, interval = "1d")

# flattening multi-level columns
raw_data.columns = [
    "_".join([str(c) for c in col if c != ""])
    if isinstance(col, tuple)
    else col
    for col in raw_data.columns
]
# creating date a column from index
raw_data = raw_data.reset_index().rename(columns={'index': 'Date'})


raw_data.head()

C:\Users\Wiltj\AppData\Local\Temp\ipykernel_11520\3071102003.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw_data = yf.download(tickers = "^GSPC", start = start_date,
[*********************100%***********************]  1 of 1 completed


,Date,Close_^GSPC,High_^GSPC,Low_^GSPC,Open_^GSPC,Volume_^GSPC
0,2017-12-18,2690.159912,2694.969971,2685.919922,2685.919922,3727770000
1,2017-12-19,2681.469971,2694.439941,2680.739990,2692.709961,3407680000
2,2017-12-20,2679.250000,2691.010010,2676.110107,2688.179932,3246230000
3,2017-12-21,2684.570068,2692.639893,2682.399902,2683.020020,3293130000
4,2017-12-22,2683.340088,2685.350098,2678.129883,2684.219971,2401030000


In [17]:
raw_data.info()
# there is no null data in the dataset
raw_data.isnull().sum()
# nothing to filter but this is where we should grab data that is ready to be used
save_path = "../../data/stock/yfinance_data_cleaned.csv"
raw_data.to_csv(save_path, index=False)
# creating a copy and saving it for better name clarity
df_stock = raw_data.copy()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 650 entries, 0 to 649
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Date          650 non-null    datetime64[ns]
 1   Close_^GSPC   650 non-null    float64       
 2   High_^GSPC    650 non-null    float64       
 3   Low_^GSPC     650 non-null    float64       
 4   Open_^GSPC    650 non-null    float64       
 5   Volume_^GSPC  650 non-null    int64         
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage: 30.6 KB


Close is the closing price of the day for the stock  
High is the highest traded price of the day for the stock  
Low is the lowest traded price of the day for the stock  
Open is the opening price of the day for the stock  
Volume is the number of shares traded during the day for the stock 

Using this information and I am going to use it to quantify percent change if the stock went up or down each day and binary classifying the data

In [14]:
# calculating percent change for closing price
df_stock['pct_change'] = df_stock['Close_^GSPC'].pct_change()

# Shifting tomorrows close
df_stock['Close_T+1'] = df_stock['Close_^GSPC'].shift(-1)

# predict whether tomorrow will go UP
df_stock['Target'] = (df_stock['Close_T+1'] > df_stock['Close_^GSPC']).astype(int)

# selecting relevant columns for our testing and training and merging into new dataframe
df_stock_refined = df_stock[['Volume_^GSPC', 'pct_change', 'Target', 'Date']].copy()

#removing the first row with NaN pct_change
df_stock_refined = df_stock_refined.iloc[1:].reset_index(drop=True)

#saving refined dataframe
save_path = "../../data/stock/yfinance_data_cleaned.csv"
df_stock_refined.to_csv(save_path, index=False)

df_stock_refined.head()

,Volume_^GSPC,pct_change,Target,Date
0,3727770000,0.005363,0,2017-12-18
1,3407680000,-0.003230,0,2017-12-19
2,3246230000,-0.000828,1,2017-12-20
3,3293130000,0.001986,0,2017-12-21
4,2401030000,-0.000458,0,2017-12-22
